In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json
import sqlite3
import pandas as pd
import re
import time

def clean_generated_sql(raw_text):
    clean = re.sub(r"```sql|```", "", raw_text)
    select_match = re.search(r"\bSELECT\b", clean, re.IGNORECASE)
    if select_match:
        clean = clean[select_match.start():]
    clean = clean.split(';')[0].split('\n\n')[0].strip()
    return clean

model_id = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit"
lora_path = "sql_model_lora" 

tokenizer = AutoTokenizer.from_pretrained(model_id)
base_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="cuda", 
    dtype=torch.float16, 
    trust_remote_code=True
)

model = PeftModel.from_pretrained(base_model, lora_path)
model.eval()

with open("tables.json", "r") as f:
    tables = json.load(f)

db_map = {}
for entry in tables:
    db_id = entry['db_id']
    schema_parts = [f"{entry['table_names_original'][c[0]]}.{c[1]}" 
                    for c in entry['column_names_original'] if c[0] != -1]
    db_map[db_id] = ", ".join(schema_parts)

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Convert the following natural language question into a valid SQL query using the provided database schema.
Schema: {}

### Input:
{}

### Response:
{}"""

with open("dev.json", "r") as f:
    dev_data = json.load(f)

results_log = []
num_to_test = 1000

print(f"Starting TRAINED (LoRA) Evaluation with Speed Tracking...")

for i in range(num_to_test):
    item = dev_data[i]
    db_id, question, gold_sql = item['db_id'], item['question'], item['query']
    db_path = f"database/{db_id}/{db_id}.sqlite"
    schema = db_map.get(db_id, "")

    prompt = prompt_template.format(schema, question, "").strip()
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    start_time = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=100, 
            do_sample=False, 
            pad_token_id=tokenizer.eos_token_id
        )
    end_time = time.time()
    
    latency = end_time - start_time
    gen_tokens = len(outputs[0]) - len(inputs.input_ids[0])
    tps = gen_tokens / latency if latency > 0 else 0

    raw_gen = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    pred_sql = clean_generated_sql(raw_gen)

    report = {
        "question": question,
        "gold": gold_sql,
        "pred": pred_sql,
        "is_executable": False,
        "exec_match": False,
        "latency": latency,
        "tps": tps
    }
    
    try:
        conn = sqlite3.connect(db_path)
        pred_df = pd.read_sql_query(pred_sql, conn)
        gold_df = pd.read_sql_query(gold_sql, conn)
        report["is_executable"] = True
        
        pred_df.columns = [c.lower() for c in pred_df.columns]
        gold_df.columns = [c.lower() for c in gold_df.columns]
        
        report["exec_match"] = pred_df.sort_values(by=list(pred_df.columns)).reset_index(drop=True).equals(
                               gold_df.sort_values(by=list(gold_df.columns)).reset_index(drop=True))
        conn.close()
    except Exception as e:
        report["error"] = str(e)
        
    results_log.append(report)
    print(f"[{i+1}/{num_to_test}] Exec: {report['is_executable']} | Match: {report['exec_match']} | Speed: {tps:.2f} t/s")

total = len(results_log)
acc = sum(1 for x in results_log if x['exec_match']) / total
valid = sum(1 for x in results_log if x['is_executable']) / total
avg_tps = sum(r['tps'] for r in results_log) / total

print(f"\n" + "="*50)
print(f"TRAINED MODEL (LORA) FINAL RESULTS")
print(f"="*50)
print(f"Execution Accuracy: {acc*100:.2f}%")
print(f"Valid SQL Rate:     {valid*100:.2f}%")
print(f"Average Speed:      {avg_tps:.2f} tokens/sec")
print(f"="*50 + "\n")

print("QUALITATIVE ANALYSIS SAMPLES:")
print("-" * 50)

for i in range(min(5, len(results_log))):
    s = results_log[i]
    print(f"Sample {i+1}:")
    print(f"  Question:  {s['question']}")
    print(f"  Generated: {s['pred']}")
    print(f"  Gold:      {s['gold']}")
    print(f"  SQL Valid: {s['is_executable']} | SQL Match: {s['exec_match']}")
    print(f"  Speed:     {s['latency']:.2f}s ({s['tps']:.2f} t/s)")
    
    if s['is_executable'] and not s['exec_match']:
        print("  Status:    Logic Error")
    elif not s['is_executable']:
        print(f"  Status:    Syntax Error ({s.get('error', 'Unknown')})")
    else:
        print("  Status:    Perfect Match")
    print("-" * 50)